In [1]:
import pandas as pd 
import numpy as np 
from pathlib import Path
from dataclasses import dataclass
import pickle
import librosa
import torchaudio
from tqdm import tqdm

/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
! hostname

node103


In [3]:
commonvoice_path = Path('/om2/user/imgriff/datasets/commonvoice_9/en/')
cv_alignments = (commonvoice_path/ 'alignment_dfs').glob('*.pdpkl')

In [4]:
## Define dataclass used to save alignments in pandas files 
@dataclass
class Alignment:
    label: str
    start: int
    end: int
    score: float

    def __repr__(self):
        return f"{self.label}\t({self.score:4.2f}): [{self.start:.4f}, {self.end:.4f})"

    @property
    def length(self):
        return self.end - self.start

    
#  fn to grab sampling rate


In [5]:
all_cv_metadata = pd.concat([pd.read_pickle(pd_path) for pd_path in cv_alignments], axis=0, ignore_index=True)

In [7]:
all_cv_metadata.to_pickle(commonvoice_path / "cv_all_alignments_pre_cut.pdpkl")

In [8]:
all_cv_metadata.head()

,path,sentence,client_id,gender,origin_index,alignment,sr
0,common_voice_en_27710027.mp3,"Joe Keaton disapproved of films, and Buster al...",000abb3006b78ea4c1144e55d9d158f05a9db011016051...,NaN,0,"[JOE\t(0.86): [0.7629, 1.0440), KEATON\t(0.86)...",32000.0
1,common_voice_en_699711.mp3,She'll be all right.,0013037a1d45cc33460806cc3f8ecee9d536c45639ba4c...,NaN,1,"[SHE'LL\t(0.62): [1.0699, 1.2717), BE\t(0.93):...",48000.0
2,common_voice_en_21953345.mp3,six,0014c5a3e5715a54855257779b89c2bb498d470b225866...,NaN,2,"[SIX\t(0.85): [1.6831, 1.9034)]",48000.0
3,common_voice_en_18132047.mp3,All's well that ends well.,001509f4624a7dee75247f6a8b642c4a0d09f8be3eeea6...,NaN,3,"[ALL'S\t(0.65): [0.7823, 1.0631), WELL\t(0.83)...",48000.0
4,common_voice_en_27340672.mp3,It is a busy market town that serves a large s...,001519f234e04528a2b36158c205dbe61c8da45ab0242f...,NaN,4,"[IT\t(0.95): [0.2208, 0.2810), IS\t(0.59): [0....",32000.0


In [13]:
all_cv_metadata.alignment[0][0].end

1.044

### Unbox alignments from sentence level manifest to word-level manifest

There are just over 1.5 million sentences in the corpora. Unfiltered word level manifest will have 14,216,574 words.

In [5]:
all_cv_metadata = pd.read_pickle(commonvoice_path / "cv_all_alignments_pre_cut.pdpkl")

In [6]:
word_level_dfs = []

for ix in tqdm(range(len(all_cv_metadata))):
    aud_path = all_cv_metadata['path'][ix]
    client_id = all_cv_metadata['client_id'][ix]
    gender = all_cv_metadata['gender'][ix]
    origin_index = all_cv_metadata['origin_index'][ix]
    alignment = all_cv_metadata['alignment'][ix]
    sr = int(all_cv_metadata['sr'][ix])
    for word_event in alignment:
        record = {"aud_path":aud_path,
                  "client_id":client_id,
                  "gender":gender,
                  "origin_index":origin_index,
                  "word": word_event.label,
                  "start_in_s": word_event.start,
                  "end_in_s": word_event.end,
                  "sr": sr
                }
        word_level_dfs.append(record)

100%|██████████| 1508500/1508500 [00:44<00:00, 33758.82it/s]


In [7]:
word_level_manifest = pd.DataFrame.from_records(word_level_dfs)
# del word_level_dfs

In [9]:
# del word_level_dfs

In [10]:
print(word_level_manifest.shape)
word_level_manifest.head()

(14216574, 8)


,aud_path,client_id,gender,origin_index,word,start_in_s,end_in_s,sr
0,common_voice_en_27710027.mp3,000abb3006b78ea4c1144e55d9d158f05a9db011016051...,NaN,0,JOE,0.762875,1.044000,32000
1,common_voice_en_27710027.mp3,000abb3006b78ea4c1144e55d9d158f05a9db011016051...,NaN,0,KEATON,1.064062,1.465563,32000
2,common_voice_en_27710027.mp3,000abb3006b78ea4c1144e55d9d158f05a9db011016051...,NaN,0,DISAPPROVED,1.505750,2.108062,32000
3,common_voice_en_27710027.mp3,000abb3006b78ea4c1144e55d9d158f05a9db011016051...,NaN,0,OF,2.188375,2.268687,32000
4,common_voice_en_27710027.mp3,000abb3006b78ea4c1144e55d9d158f05a9db011016051...,NaN,0,FILMS,2.369063,2.710375,32000


In [11]:
word_level_manifest.to_pickle(commonvoice_path / "cv_all_word_alignments.pdpkl")

In [12]:
word_level_manifest.sr.unique()

array([32000, 48000, 44100, 16000])

In [ ]:
# word_level_manifest = pd.read_pickle(commonvoice_path / "cv_all_word_alignments.pdpkl")

In [ ]:
# from joblib import Parallel, delayed

In [ ]:
# paths = word_level_manifest.aud_path.unique().tolist()

# def get_sr(path):
#     _, sr = torchaudio.load(commonvoice_path / 'clips' / path)
#     return {path:sr}

# task = (delayed(get_sr)(path) for path in tqdm(paths))
# srs = Parallel(20)(task)

100%|██████████| 1338110/1338110 [21:56<00:00, 1016.26it/s]


In [ ]:
# file_srs = {}
# for pair in srs:
#     file_srs.update(pair)
    
# word_level_manifest['sr'] = word_level_manifest['aud_path'].map(file_srs)

In [13]:
word_level_manifest

,aud_path,client_id,gender,origin_index,word,start_in_s,end_in_s,sr
0,common_voice_en_27710027.mp3,000abb3006b78ea4c1144e55d9d158f05a9db011016051...,NaN,0,JOE,0.762875,1.044000,32000
1,common_voice_en_27710027.mp3,000abb3006b78ea4c1144e55d9d158f05a9db011016051...,NaN,0,KEATON,1.064062,1.465563,32000
2,common_voice_en_27710027.mp3,000abb3006b78ea4c1144e55d9d158f05a9db011016051...,NaN,0,DISAPPROVED,1.505750,2.108062,32000
3,common_voice_en_27710027.mp3,000abb3006b78ea4c1144e55d9d158f05a9db011016051...,NaN,0,OF,2.188375,2.268687,32000
4,common_voice_en_27710027.mp3,000abb3006b78ea4c1144e55d9d158f05a9db011016051...,NaN,0,FILMS,2.369063,2.710375,32000
...,...,...,...,...,...,...,...,...
14216569,common_voice_en_31747838.mp3,372293e65cdab88771e028a4351651ab2eff64438ddafc...,male,1556253,TIDAL,2.724937,3.506375,32000
14216570,common_voice_en_31747838.mp3,372293e65cdab88771e028a4351651ab2eff64438ddafc...,male,1556253,AND,3.626625,3.746812,32000
14216571,common_voice_en_31747838.mp3,372293e65cdab88771e028a4351651ab2eff64438ddafc...,male,1556253,TOWER,4.027375,4.568313,32000
14216572,common_voice_en_31747838.mp3,372293e65cdab88771e028a4351651ab2eff64438ddafc...,male,1556253,RECORDS,4.688562,5.249562,32000


In [14]:
word_level_manifest.sr.value_counts()

48000    10803078
32000     3284980
44100      128453
16000          63
Name: sr, dtype: int64

## Following is old/exploratory 
Get valid vocabulary

In [15]:
word_and_speaker_encodings = pickle.load( open( "/om2/user/imgriff/projects/Auditory-Attention/word_and_speaker_encodings_jsinv3.pckl", "rb" )) 
class_map = word_and_speaker_encodings['word_idx_to_word']

vocab = [word.upper() for word in class_map.values() if '__null' not in word]  # [1:] to cut '__nullSignal__'


In [16]:
word_level_manifest_wsn_vocab = word_level_manifest[word_level_manifest.word.isin(vocab)]

In [17]:
word_level_manifest_wsn_vocab.sr.value_counts()

48000    1610915
32000     560463
44100      22387
16000          7
Name: sr, dtype: int64

## Cut based on dataset split

In [ ]:
!ls {commonvoice_path}

In [ ]:
get_sr = lambda x:torchaudio.load(commonvoice_path / 'clips' / x)[1]



In [ ]:
test_manifest = pd.read_csv(commonvoice_path / 'test.tsv', sep='\t' )

test_names = test_manifest['client_id']

In [ ]:
test_word_level_manifest = word_level_manifest[word_level_manifest.client_id.isin(test_names)]

In [ ]:
# test_word_level_manifest_w_gender = test_word_level_manifest[test_word_level_manifest.gender.isin(['male', 'female'])]


In [ ]:
test_word_level_manifest_w_words = test_word_level_manifest[test_word_level_manifest.word.isin(vocab)]
test_word_level_manifest_w_words = test_word_level_manifest_w_words.reset_index(drop=True)


In [ ]:
test_word_level_manifest_w_words.head()

In [ ]:
test_word_level_manifest_w_words['sr'] = [get_sr(path) for path in tqdm(test_word_level_manifest_w_words['aud_path'])]

In [ ]:
test_word_level_manifest_w_words.head()

In [ ]:
test_word_level_manifest_w_words.sr.value_counts()

In [ ]:
test_word_level_manifest_w_words.to_pickle(commonvoice_path / "cv_test_set_wsn_vocab_word_alignments.pdpkl")

### gender balence test set

In [ ]:
test_word_level_manifest_w_words_gender = test_word_level_manifest_w_words[test_word_level_manifest_w_words.gender != 'other']

In [ ]:
test_word_level_manifest_w_words_gender.gender.value_counts()

In [ ]:
np.random.seed(0)
test_word_level_manifest_w_words_gender = test_word_level_manifest_w_words_gender.groupby('gender').sample(n=625).reset_index(drop=True)# is N of female
test_word_level_manifest_w_words_gender

In [ ]:
test_word_level_manifest_w_words_gender.sr.unique()

In [ ]:
test_word_level_manifest_w_words_gender.shape

In [ ]:
test_word_level_manifest_w_words_gender.to_pickle(commonvoice_path / "cv_test_set_wsn_vocab_word_alignments_gender_balanced_all_sr.pdpkl")

In [ ]:
test_at_48 = test_word_level_manifest_w_words_gender[test_word_level_manifest_w_words_gender.sr==48000]

In [ ]:
torchaudio.load(commonvoice_path / 'clips' / 'common_voice_en_20498760.mp3')[1]

In [ ]:
test_at_48.to_pickle(commonvoice_path / "cv_test_set_wsn_vocab_word_alignments_gender_balanced_48kHz_sr.pdpkl")